# M2 · Feature engineering & leakage

_Curriculum · Domain 0 · ML Foundations_

**Build leak-free features by respecting the prediction-time clock.**

We create a tiny ads click dataset with one honest historical feature and one leaky future feature. Run each cell top to bottom. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2)

## The point-in-time rule

For a row scored at time $t_i$, valid aggregates use only prior events:

$$c_i = \sum_j \mathbf{1}\{\text{event}_j < t_i\}$$

A feature computed after $t_i$ can accidentally contain the label.

In [ ]:
n = 3000
past_clicks = rng.poisson(2.0, size=n)
country = rng.choice(["US", "IN", "BR", "DE"], size=n, p=[0.55, 0.25, 0.12, 0.08])
country_boost = np.where(country == "US", 0.25, 0.0)
logit = -3.2 + 0.35 * past_clicks + country_boost
p = 1.0 / (1.0 + np.exp(-logit))
clicked = (rng.random(n) < p).astype(int)
future_click_signal = clicked + rng.binomial(1, 0.03, size=n)

df = pd.DataFrame({"past_clicks": past_clicks, "country": country, "future_click_signal": future_click_signal, "clicked": clicked})
df.head()

## Step 1 - Look at the suspicious feature

A future signal should look too predictive because it is measured after the outcome window begins.

In [ ]:
rate_by_future = df.groupby("future_click_signal")["clicked"].mean()

print(rate_by_future)

assert rate_by_future.loc[1] > rate_by_future.loc[0] + 0.5

## Step 2 - Build an honest feature matrix

We keep `past_clicks` and one-hot encode country. We do not include the future signal.

In [ ]:
X_honest = pd.get_dummies(df[["past_clicks", "country"]], columns=["country"], drop_first=True)
y = df["clicked"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(X_honest, y, test_size=0.3, random_state=2, stratify=y)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_train_scaled[["past_clicks"]] = scaler.fit_transform(X_train[["past_clicks"]])
X_val_scaled[["past_clicks"]] = scaler.transform(X_val[["past_clicks"]])

assert abs(X_train_scaled["past_clicks"].mean()) < 1e-12

## Step 3 - Compare honest and leaky validation AUC (area under the ROC curve)

The leaky model gets an offline score that would not survive serving.

In [ ]:
honest_model = LogisticRegression(max_iter=1000)
honest_model.fit(X_train_scaled, y_train)

p_honest = honest_model.predict_proba(X_val_scaled)[:, 1]
auc_honest = roc_auc_score(y_val, p_honest)

X_leaky = X_honest.copy()
X_leaky["future_click_signal"] = df["future_click_signal"]
Xl_train, Xl_val, yl_train, yl_val = train_test_split(X_leaky, y, test_size=0.3, random_state=2, stratify=y)
leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(Xl_train, yl_train)
p_leaky = leaky_model.predict_proba(Xl_val)[:, 1]
auc_leaky = roc_auc_score(yl_val, p_leaky)

print("honest AUC", round(auc_honest, 3))
print("leaky AUC", round(auc_leaky, 3))

assert auc_leaky > auc_honest + 0.2

## Visualize the leakage jump

A giant offline improvement from one availability-violating feature is a leakage smell, not a launch plan.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["honest", "leaky"], [auc_honest, auc_leaky], color=["#4c78a8", "#e45756"])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("validation AUC")
ax.set_title("leakage inflates offline metrics")
plt.show()

## Practice

1. Replace `past_clicks` with `np.log1p(past_clicks)` and compare AUC.
2. Add a rare country bucket before one-hot encoding.
3. Write a sentence explaining why train-only scaling matters.

In [ ]:
# Your turn:
